In [2]:
import os
from pathlib import Path
import mlflow
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb

In [5]:
%pwd

print("Changing directory")
os.chdir("../")
%pwd

Changing directory


'/Users/harendrakumar/Documents/Loan_Prediction'

In [30]:
df = pd.read_csv("artifacts/data_ingestion/credit_train.csv")
df.head(2)

,Loan ID,Customer ID,Loan Status,Current Loan Amount,Term,Credit Score,Annual Income,Years in current job,Home Ownership,Purpose,Monthly Debt,Years of Credit History,Months since last delinquent,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens
0,14dd8831-6af5-400b-83ec-68e61888a048,981165ec-3274-42f5-a3b4-d104041a9ca9,Fully Paid,445412.0,Short Term,709.0,1167493.0,8 years,Home Mortgage,Home Improvements,5214.74,17.2,NaN,6.0,1.0,228190.0,416746.0,1.0,0.0
1,4771cc26-131a-45db-b5aa-537ea4ba5342,2de017a3-2e01-49cb-a581-08169e83be29,Fully Paid,262328.0,Short Term,NaN,NaN,10+ years,Home Mortgage,Debt Consolidation,33295.98,21.1,8.0,35.0,0.0,229976.0,850784.0,0.0,0.0


In [31]:
df.isnull().sum()

Loan ID                           514
Customer ID                       514
Loan Status                       514
Current Loan Amount               514
Term                              514
Credit Score                    19668
Annual Income                   19668
Years in current job             4736
Home Ownership                    514
Purpose                           514
Monthly Debt                      514
Years of Credit History           514
Months since last delinquent    53655
Number of Open Accounts           514
Number of Credit Problems         514
Current Credit Balance            514
Maximum Open Credit               516
Bankruptcies                      718
Tax Liens                         524
dtype: int64

In [32]:
df.dropna(inplace=True)

In [33]:
df.isnull().sum()

Loan ID                         0
Customer ID                     0
Loan Status                     0
Current Loan Amount             0
Term                            0
Credit Score                    0
Annual Income                   0
Years in current job            0
Home Ownership                  0
Purpose                         0
Monthly Debt                    0
Years of Credit History         0
Months since last delinquent    0
Number of Open Accounts         0
Number of Credit Problems       0
Current Credit Balance          0
Maximum Open Credit             0
Bankruptcies                    0
Tax Liens                       0
dtype: int64

In [34]:
df.shape

(36423, 19)

In [36]:
df.drop(["Loan ID", "Customer ID"], axis=1, inplace=True)

In [50]:
cat_cols = df.columns[df.dtypes=="object"]
cat_cols

Index(['Term', 'Years in current job', 'Home Ownership', 'Purpose'], dtype='object')

In [29]:
from sklearn.preprocessing import LabelEncoder

In [55]:
def encode_categorical_values(df: pd.DataFrame) -> pd.DataFrame:
    cat_col = df.select_dtypes(include="object").columns
    num_cols = df.select_dtypes(include=["int", "float"]).columns

    num_data = df[num_cols].copy()
    
    le = LabelEncoder()
    encoded_data = df[cat_col].apply(le.fit_transform)
    
    complete_data = pd.concat([num_data, encoded_data], axis=1)
    return complete_data

In [39]:
mapping = {
    "Charged Off": 0,
    "Fully Paid": 1
}

In [40]:
df["Loan Status"] = df["Loan Status"].map(mapping)

In [56]:
tranformed_data = encode_categorical_values(df=df)

In [57]:
tranformed_data.head(2)

,Loan Status,Current Loan Amount,Credit Score,Annual Income,Monthly Debt,Years of Credit History,Months since last delinquent,Number of Open Accounts,Number of Credit Problems,Current Credit Balance,Maximum Open Credit,Bankruptcies,Tax Liens,Term,Years in current job,Home Ownership,Purpose
2,1,99999999.0,741.0,2231892.0,29200.53,14.9,29.0,18.0,1.0,297996.0,750090.0,0.0,0.0,1,8,2,3
6,1,217646.0,730.0,1184194.0,10855.08,19.6,10.0,13.0,1.0,122170.0,272052.0,1.0,0.0,1,10,1,3


In [13]:
from sklearn.model_selection import train_test_split

In [68]:
X = tranformed_data.drop("Loan Status", axis=1)
y = tranformed_data["Loan Status"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

In [69]:
model = xgb.XGBClassifier(n_estimators=50, max_depth=500, max_leaves=200, )

In [70]:
model.fit(X_train, y_train)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [71]:
y_pred = model.predict(X_test)

In [72]:
from sklearn.metrics import f1_score, accuracy_score,roc_auc_score, roc_curve

In [75]:
np.array(y_test)

array([0, 0, 1, ..., 1, 1, 1])

In [77]:
f1_score(y_test, y_pred)

0.9043052963511166

In [78]:
accuracy_score(y_test, y_pred)

0.8367010762134857

In [82]:
y_score = model.predict_proba(X_test)[:, 1]
y_score

array([0.8124171 , 0.93770313, 0.6627697 , ..., 0.9832592 , 0.99977833,
       0.9280302 ], dtype=float32)

In [83]:
roc_auc_score(y_test, y_score)

0.7722774023415561

In [85]:
roc_curve(y_test, y_score)

(array([0.        , 0.        , 0.        , ..., 0.75461741, 0.75461741,
        1.        ]),
 array([0.00000000e+00, 1.38677021e-04, 1.94147830e-03, ...,
        9.99861323e-01, 1.00000000e+00, 1.00000000e+00]),
 array([          inf, 9.9997008e-01, 9.9994922e-01, ..., 8.4919341e-02,
        4.5019723e-02, 6.7467059e-05], dtype=float32))

In [87]:
from src.mlProject.utils.optimization import HyperparameterTuning

In [88]:
n_estimators = [5,21,51,101] # number of trees in the random forest
max_features = ['auto', 'sqrt'] # number of features in consideration at every split
max_depth = [int(x) for x in np.linspace(10, 120, num = 12)] # maximum number of levels allowed in each decision tree
min_samples_split = [2, 6, 10] # minimum sample number to split a node
min_samples_leaf = [1, 3, 4] # minimum sample number that can be stored in a leaf node
bootstrap = [True, False]

In [89]:
optimize = HyperparameterTuning(n_estimators=n_estimators,
                               max_features=max_features,
                               max_depth=max_depth,
                               min_samples_split=min_samples_split,
                               min_samples_leaf=min_samples_split,
                               bootstrap=bootstrap)

In [91]:
optimize.optimize(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


/Users/harendrakumar/.local/share/virtualenvs/harendrakumar-Fk8KTnIY/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
10 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/harendrakumar/.local/share/virtualenvs/harendrakumar-Fk8KTnIY/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/harendrakumar/.local/share/virtualenvs/harendrakumar-Fk8KTnIY/lib/python3.13/site-packages/sklearn/base.py", line 1358, in wrapper
    

[2025-11-19 22:41:06,359: INFO: optimization: Random Grid: {'n_estimators': [5, 21, 51, 101], 'max_features': ['auto', 'sqrt'], 'max_depth': [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120], 'min_samples_split': [2, 6, 10], 'min_samples_leaf': [2, 6, 10], 'bootstrap': [True, False]}]
[2025-11-19 22:41:06,361: INFO: optimization: best_params: {'n_estimators': 51, 'min_samples_split': 2, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'max_depth': 40, 'bootstrap': False}]


{'n_estimators': 51,
 'min_samples_split': 2,
 'min_samples_leaf': 6,
 'max_features': 'sqrt',
 'max_depth': 40,
 'bootstrap': False}